In [2]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery

In [5]:
CREDS = "../../converge-database-0331482f2ee5.json"
project_id = "converge-database"
Target_table_D = "All_Contracts_FIA.All_Contracts_DENALI"
Target_table_T = "All_Contracts_FIA.All_Contracts_TETON"
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [6]:
query_policy = """
SELECT *
FROM `converge-database.denali.policy`
"""
policy_df = client.query(query_policy).to_dataframe()

query_seriatim = """
SELECT *
FROM `converge-database.denali.seriatim_values`
"""
seriatim_df = client.query(query_seriatim).to_dataframe()

query_commissions = """
SELECT *
FROM `converge-database.denali.commissions`
"""
commissions_df = client.query(query_commissions).to_dataframe()

query_deaths = """
SELECT *
FROM `converge-database.denali.deaths`
"""
deaths_df = client.query(query_deaths).to_dataframe()

query_withdrawals = """
SELECT *
FROM `converge-database.denali.withdrawals`
"""
withdrawals_df = client.query(query_withdrawals).to_dataframe()


KeyboardInterrupt



In [ ]:
policy_df["issueyear"] = pd.to_numeric(policy_df["issueyear"], errors="coerce")
policy_df["issuemonth"] = pd.to_numeric(policy_df["issuemonth"], errors="coerce")
policy_df["issueday"] = pd.to_numeric(policy_df["issueday"], errors="coerce")

policy_df["issue_date"] = pd.to_datetime(
    policy_df["issueyear"].fillna(0).astype(int).astype(str).str.zfill(4) +
    policy_df["issuemonth"].fillna(0).astype(int).astype(str).str.zfill(2) +
    policy_df["issueday"].fillna(0).astype(int).astype(str).str.zfill(2),
    format="%Y%m%d",
    errors="coerce"
)

In [ ]:
# fields from `denali.policy`

policy_use = policy_df[[
    "policynumber",
    "product",
    "plan",
    "issueyear",
    "issuemonth",
    "issueday",
    "issue_date",
    "issueage"
]].copy()

In [ ]:
# fields from `denali.seriatim`

seriatim_df["totalinitpremium"] = pd.to_numeric(seriatim_df["totalinitpremium"], errors="coerce")
seriatim_df["totalpolicyiav"] = pd.to_numeric(seriatim_df["totalpolicyiav"], errors="coerce")
seriatim_df["converge"] = pd.to_numeric(seriatim_df["converge"], errors="coerce")

seriatim_use = seriatim_df[[
    "policynumber",
    "totalinitpremium",
    "totalpolicyiav",
    "converge"
]].copy()

In [ ]:
# fields from `denali.commissions`

commissions_df["approvaldate"] = pd.to_datetime(commissions_df["approvaldate"], errors="coerce")
commissions_use = commissions_df[[
    "policynumber",
    "approvaldate"
]].copy()
commissions_use = commissions_use.rename(columns={
    "approvaldate": "date_approved"
})

In [ ]:
# fields from `denali.deathclaims`

deaths_df["deathclaims"] = pd.to_numeric(deaths_df["deathclaims"], errors="coerce")
deaths_df["termdate"] = pd.to_datetime(deaths_df["termdate"], errors="coerce")

death_use = deaths_df.loc[
    deaths_df["deathclaims"].fillna(0) != 0,
    ["policynumber", "termdate"]
].copy()
death_use = death_use.rename(columns={
    "termdate": "death_termdate"
})

In [ ]:
# fields from `denali.withdrawal tables`

withdrawals_df["fullsurrenders"] = pd.to_numeric(withdrawals_df["fullsurrenders"], errors="coerce")
withdrawals_df["termdate"] = pd.to_datetime(withdrawals_df["termdate"], errors="coerce")

withdrawals_use = withdrawals_df.loc[
    withdrawals_df["fullsurrenders"].fillna(0) != 0,
    ["policynumber", "termdate"]
].copy()

withdrawals_use = withdrawals_use.rename(columns={
    "termdate": "full_surrender_termdate"
})

In [ ]:
std_seriatim = policy_use.merge(
    seriatim_use,
    on="policynumber",
    how="left"
)

std_seriatim = std_seriatim.merge(
    commissions_use,
    on="policynumber",
    how="left"
)

std_seriatim = std_seriatim.merge(
    death_use,
    on="policynumber",
    how="left"
)

std_seriatim = std_seriatim.merge(
    withdrawals_use,
    on="policynumber",
    how="left"
)

std_seriatim["surrender_date"] = std_seriatim["death_termdate"]
std_seriatim.loc[
    std_seriatim["surrender_date"].isna(),
    "surrender_date"
] = std_seriatim["full_surrender_termdate"]

std_seriatim["termination_type"] = None
std_seriatim.loc[
    std_seriatim["death_termdate"].notna(),
    "termination_type"
] = "death"

std_seriatim.loc[
    (std_seriatim["death_termdate"].isna()) &
    (std_seriatim["full_surrender_termdate"].notna()),
    "termination_type"
] = "full surrender"

rate = std_seriatim[[
    "policynumber",
    "product",
    "plan",
    "issueyear",
    "issuemonth",
    "issueday",
    "issue_date",
    "issueage",
    "totalinitpremium",
    "totalpolicyiav",
    "converge",
    "date_approved",
    "surrender_date",
    "termination_type"
]].copy()

Target_table = Target_table_D

rate.to_gbq(
    destination_table=Target_table,
    project_id=project_id,
    if_exists="replace",
    credentials=service_account.Credentials.from_service_account_file(CREDS)
)

In [ ]:
#Seriatim and Transactions start here 

In [ ]:
Target_table_D_S = "Data_Quality_FIA.Std_denali_seriatim"
Target_table_D_T = "Data_Quality_FIA.Std_denali_trans"
Target_table_T_S = "Data_Quality_FIA.Std_teton_seriatim"
Target_table_T_T = "Data_Quality_FIA.Std_teton_trans"

In [ ]:
def compare_fix_snapshot_to_existing(fix_snapshot_df, fix_existing_df):
    compare_cols = [
        "product",
        "plan",
        "term",
        "issueyear",
        "issuemonth",
        "issueday",
        "issue_date",
        "issueage",
        "date_approved",
        "surrender_date",
        "termination_type"
    ]

    fix_snapshot_df = fix_snapshot_df.copy()
    fix_existing_df = fix_existing_df.copy()

    fix_snapshot_df = keep_first_by_key(fix_snapshot_df, ["policynumber"])
    fix_existing_df = keep_first_by_key(fix_existing_df, ["policynumber"])

    merged = fix_existing_df.merge(
        fix_snapshot_df,
        on="policynumber",
        how="outer",
        suffixes=("_old", "_new"),
        indicator=True
    )

    # new policies: only in new snapshot
    new_policy_df = merged.loc[
        merged["_merge"] == "right_only",
        ["policynumber"] + [f"{c}_new" for c in compare_cols]
    ].copy()
    new_policy_df.columns = ["policynumber"] + compare_cols

    # existing policies found in both
    existing_df = merged.loc[merged["_merge"] == "both"].copy()

    diff_cols = []
    for c in compare_cols:
        diff_col = f"diff_{c}"
        existing_df[diff_col] = (
            existing_df[f"{c}_old"].fillna("<<NULL>>").astype(str) !=
            existing_df[f"{c}_new"].fillna("<<NULL>>").astype(str)
        )
        diff_cols.append(diff_col)

    existing_df["diff_count"] = existing_df[diff_cols].sum(axis=1)

    matched_df = existing_df.loc[existing_df["diff_count"] == 0].copy()
    unmatched_df = existing_df.loc[existing_df["diff_count"] > 0].copy()

    return matched_df, unmatched_df, new_policy_df